# Go Ananas! — Dataset Building and Model Training (Google Colab)

This notebook builds the image dataset from **open, CC BY 4.0 licensed** pineapple datasets, trains the MobileNetV2
model described in Chapter 3, evaluates it, and exports the files used by the Android app and the web application.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU`.
2. Create a free account at **roboflow.com**, then open *Settings → API Keys* and copy your **Private API Key**.
3. (Optional, more images) Download these Mendeley Data datasets with the **Download All** button and upload the zip files
   to your Google Drive folder `GoAnanas/mendeley/`:
   * Pineapple Leaf Disease Dataset (Malik et al., 2026) — https://data.mendeley.com/datasets/gc5f8dfgrt/2
   * PineappleLeaf-Fruit Dataset (Islam, 2026) — https://data.mendeley.com/datasets/tv7zfd6yzz/1
4. Run the cells from top to bottom (`Runtime → Run all`). Training takes about 20–40 minutes on a T4 GPU.

**Classes** (same as the paper) plus a rejection class:

| Folder | Condition | Sources used |
|---|---|---|
| `fruit_rot` | Fruit rot | Roboflow pineapple-diseases (Fruit_Rot), PinyaSuri AI 2 (Fruit Rot Disease) |
| `healthy` | Healthy | Roboflow (No_Disease, Healthy), Mendeley (Healthy Pineapple Leaf, Healthy Leaf, Fruit) |
| `heart_rot` | Heart rot (also called crown or top rot) | Roboflow (Crown_Rot_Disease), PinyaSuri AI 2 (Crown Rot Disease) |
| `mealybug_wilt` | Mealybug wilt | Roboflow (Mealybug_Wilt), PinyaSuri AI 2, Mendeley (Pineapple Mealybug Wilt) |
| `multiple_crown_disorder` | Multiple crown disorder / fasciation | Roboflow (Multiple_Crown_Disorder, Fruit_Fasciation_Disorder), PinyaSuri AI 2 |
| `root_rot` | Root rot | Roboflow (Root_Rot) |
| `not_pineapple` | Not a pineapple (rejection) | PlantVillage leaves of other crops + Imagenette everyday objects (TensorFlow Datasets) |

In [ ]:
!pip -q install roboflow imagehash opencv-python-headless tensorflow-datasets
from google.colab import drive
drive.mount('/content/drive')
import os, json, csv, glob, shutil, random, zipfile, hashlib, re
from pathlib import Path
ROOT = Path('/content/go_ananas'); RAW = ROOT/'raw'; ALL = ROOT/'dataset'/'all'
for p in (RAW, ALL): p.mkdir(parents=True, exist_ok=True)
DRIVE = Path('/content/drive/MyDrive/GoAnanas'); DRIVE.mkdir(parents=True, exist_ok=True)
CLASSES = ['fruit_rot', 'healthy', 'heart_rot', 'mealybug_wilt', 'multiple_crown_disorder', 'not_pineapple', 'root_rot']
MAX_PER_CLASS = 300      # the paper uses 300 images per class; raise it (e.g. 600) to use more images when available
SEED = 42
random.seed(SEED)
for c in CLASSES: (ALL/c).mkdir(exist_ok=True)

## 1. Download the Roboflow datasets (CC BY 4.0)

In [ ]:
ROBOFLOW_API_KEY = ''   # <-- paste your Roboflow Private API Key here
assert ROBOFLOW_API_KEY, 'Paste your Roboflow API key in ROBOFLOW_API_KEY first.'
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
PROJECTS = [
    ('pineapple-diseases', 'dataset-50-yd482'),
    ('pineapple-diseases', 'dataset-340-kyezk'),
    ('pineapple-diseases', 'dataset-1050-z2ebe'),
    ('ece-4-workspace', 'pinyasuri-ai-2'),
]
downloaded = []
for ws, proj in PROJECTS:
    try:
        p = rf.workspace(ws).project(proj)
        versions = p.versions()
        v = versions[0] if versions else None
        if v is None:
            print('no versions:', proj); continue
        ptype = getattr(p, 'type', '')
        fmts = ['folder', 'multiclass'] if 'classification' in str(ptype) else ['coco', 'folder', 'multiclass']
        for fmt in fmts:
            try:
                out = RAW/f'{proj}_{fmt}'
                v.download(fmt, location=str(out), overwrite=True)
                downloaded.append((proj, fmt, out)); print('OK', proj, fmt); break
            except Exception as e:
                print('  format', fmt, 'failed for', proj, '->', e)
    except Exception as e:
        print('Could not open', ws, proj, '->', e)
print(downloaded)

## 2. Map the source labels to the seven classes

In [ ]:
def norm(s): return re.sub(r'[^a-z]', '', s.lower())
LABEL_MAP = {
    'fruitrot': 'fruit_rot', 'fruitrotdisease': 'fruit_rot',
    'nodisease': 'healthy', 'healthy': 'healthy', 'healthypineappleleaf': 'healthy', 'healthyleaf': 'healthy', 'fruit': 'healthy',
    'crownrot': 'heart_rot', 'crownrotdisease': 'heart_rot', 'heartrot': 'heart_rot', 'toprot': 'heart_rot',
    'mealybugwilt': 'mealybug_wilt', 'mealybugwiltdisease': 'mealybug_wilt', 'pineapplemealybugwilt': 'mealybug_wilt',
    'multiplecrowndisorder': 'multiple_crown_disorder', 'fruitfasciationdisorder': 'multiple_crown_disorder', 'fasciation': 'multiple_crown_disorder',
    'rootrot': 'root_rot', 'rootrotdisease': 'root_rot',
}
def map_label(name):
    return LABEL_MAP.get(norm(name))

collected = {c: [] for c in CLASSES}     # class -> list of (path_or_crop_callable, source)
import cv2
CROPS = ROOT/'crops'; CROPS.mkdir(exist_ok=True)

def add_file(cls, path, source):
    if cls: collected[cls].append((str(path), source))

for proj, fmt, out in downloaded:
    out = Path(out)
    if fmt == 'folder':
        for f in out.rglob('*'):
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                add_file(map_label(f.parent.name), f, proj)
    elif fmt == 'multiclass':
        for csvf in out.rglob('_classes.csv'):
            rows = list(csv.reader(open(csvf)))
            head = [h.strip() for h in rows[0]]
            for r in rows[1:]:
                labels = [head[i] for i in range(1, len(head)) if r[i].strip() == '1']
                mapped = {map_label(l) for l in labels} - {None}
                if len(mapped) == 1:                     # keep images with one clear label only
                    add_file(mapped.pop(), csvf.parent/r[0], proj)
    elif fmt == 'coco':
        for js in out.rglob('_annotations.coco.json'):
            data = json.load(open(js))
            cats = {c['id']: c['name'] for c in data['categories']}
            imgs = {i['id']: i for i in data['images']}
            for a in data['annotations']:
                cls = map_label(cats.get(a['category_id'], ''))
                if not cls: continue
                info = imgs[a['image_id']]; src = js.parent/info['file_name']
                im = cv2.imread(str(src))
                if im is None: continue
                x, y, w, h = [int(v) for v in a['bbox']]
                if w < 40 or h < 40: continue
                pad = int(0.1 * max(w, h))
                crop = im[max(0, y-pad):y+h+pad, max(0, x-pad):x+w+pad]
                dst = CROPS/f"{proj}_{a['id']}.jpg"; cv2.imwrite(str(dst), crop)
                add_file(cls, dst, proj)
for c in CLASSES: print(f'{c:26s}', len(collected[c]))

## 3. (Optional) Add the Mendeley datasets from Google Drive

In [ ]:
MENDELEY = DRIVE/'mendeley'
for z in sorted(MENDELEY.glob('*.zip')) if MENDELEY.exists() else []:
    dst = RAW/('mendeley_' + z.stem); dst.mkdir(exist_ok=True)
    with zipfile.ZipFile(z) as f: f.extractall(dst)
    for inner in list(dst.rglob('*.zip')):            # some Mendeley downloads contain zips inside
        with zipfile.ZipFile(inner) as f: f.extractall(inner.parent/inner.stem)
    for f in dst.rglob('*'):
        if f.suffix.lower() in ('.jpg', '.jpeg', '.png') and 'augment' not in str(f).lower():
            add_file(map_label(f.parent.name), f, 'mendeley_' + z.stem)
for c in CLASSES: print(f'{c:26s}', len(collected[c]))

## 4. Add the *not pineapple* class (for the rejection message in the app)

In [ ]:
import tensorflow_datasets as tfds, numpy as np
def take(name, split, n, prefix):
    ds = tfds.load(name, split=split, shuffle_files=True, as_supervised=True)
    i = 0
    for img, _ in ds.take(n):
        p = CROPS/f'{prefix}_{i}.jpg'; cv2.imwrite(str(p), cv2.cvtColor(img.numpy(), cv2.COLOR_RGB2BGR)); i += 1
        add_file('not_pineapple', p, name)
take('plant_village', 'train', MAX_PER_CLASS // 2, 'pv')         # leaves of other crops
take('imagenette/160px', 'train', MAX_PER_CLASS // 2, 'inet')    # everyday objects, people, animals
print('not_pineapple', len(collected['not_pineapple']))

## 5. Remove duplicates, balance the classes, and copy to `dataset/all/`

In [ ]:
from PIL import Image
import imagehash
summary = []
for c in CLASSES:
    seen, keep = [], []
    items = collected[c][:]; random.shuffle(items)
    for path, src in items:
        try:
            h = imagehash.phash(Image.open(path).convert('RGB'))
        except Exception:
            continue
        if any(h - s <= 4 for s in seen):                 # near-duplicate (e.g. Roboflow augmented copy)
            continue
        seen.append(h); keep.append((path, src))
        if len(keep) >= MAX_PER_CLASS: break
    for i, (path, src) in enumerate(keep):
        Image.open(path).convert('RGB').save(ALL/c/f'{c}_{i:04d}.jpg', quality=92)
    by_src = {}
    for _, s in keep: by_src[s] = by_src.get(s, 0) + 1
    summary.append((c, len(keep), by_src))
    flag = '' if len(keep) >= MAX_PER_CLASS else '   <-- fewer than MAX_PER_CLASS; add more images for this class'
    print(f'{c:26s} {len(keep):5d}  {by_src}{flag}')
json.dump(summary, open(DRIVE/'dataset_summary.json', 'w'), indent=1)

## 6. Split 70 / 15 / 15 and train (Table 3.5 settings)

In [ ]:
%%writefile /content/split_dataset.py
"""Split dataset/all/<class>/ into dataset/train, dataset/val, dataset/test (70/15/15).
With 300 photos per class this gives 210 / 45 / 45 per class (270 test images in total)."""
import argparse
import random
import shutil
from pathlib import Path

ap = argparse.ArgumentParser()
ap.add_argument("--src", default="dataset/all")
ap.add_argument("--dst", default="dataset")
ap.add_argument("--seed", type=int, default=42)
args = ap.parse_args()

random.seed(args.seed)
for cls in sorted(p for p in Path(args.src).iterdir() if p.is_dir()):
    files = sorted(f for f in cls.iterdir() if f.is_file())
    random.shuffle(files)
    n = len(files)
    n_tr, n_va = round(n * 0.70), round(n * 0.15)
    parts = {"train": files[:n_tr], "val": files[n_tr:n_tr + n_va], "test": files[n_tr + n_va:]}
    for split, items in parts.items():
        d = Path(args.dst) / split / cls.name
        d.mkdir(parents=True, exist_ok=True)
        for f in items:
            shutil.copy2(f, d / f.name)
    print(f"{cls.name}: train {len(parts['train'])}, val {len(parts['val'])}, test {len(parts['test'])}")

In [ ]:
%%writefile /content/train_model.py
"""
Go Ananas! model training script.

Trains the MobileNetV2 classifier described in Chapter 3 (Table 3.5) and exports
it to TensorFlow Lite for the Android app and the web app.

Dataset layout (one folder per class, names in lowercase with underscores):

    dataset/
      train/  fruit_rot/ healthy/ heart_rot/ mealybug_wilt/ multiple_crown_disorder/ not_pineapple/ root_rot/
      val/    (same six folders)
      test/   (same six folders)

If only dataset/all/<class>/ exists, run split_dataset.py first (70/15/15 split).

Usage:
    python train_model.py --data dataset --out output --copy-to-app

Outputs (in --out):
    go_ananas_model.keras, go_ananas_model.tflite, labels.txt, model_info.json,
    confusion_matrix.csv, classification_report.txt, training_history.csv
"""
import argparse
import csv
import json
import shutil
from datetime import date
from pathlib import Path

import cv2
import numpy as np
import tensorflow as tf

IMG = 224
# six conditions from the paper plus "not_pineapple", which lets the app reject photos that do not show a pineapple
CLASSES = ["fruit_rot", "healthy", "heart_rot", "mealybug_wilt", "multiple_crown_disorder", "not_pineapple", "root_rot"]
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def preprocess(path: str) -> np.ndarray:
    """Same steps as the Android app (ImagePreprocessor.kt) and the web app:
    resize 224x224 -> Gaussian blur 3x3 -> CLAHE on L channel of LAB -> scale to [-1, 1]."""
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise ValueError(f"Cannot read {path}")
    bgr = cv2.resize(bgr, (IMG, IMG), interpolation=cv2.INTER_AREA)
    bgr = cv2.GaussianBlur(bgr, (3, 3), 0)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    rgb = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)
    return rgb.astype(np.float32) / 127.5 - 1.0


def load_split(root: Path):
    xs, ys = [], []
    for idx, name in enumerate(CLASSES):
        folder = root / name
        if not folder.is_dir():
            raise SystemExit(f"Missing folder: {folder}")
        for f in sorted(folder.iterdir()):
            if f.suffix.lower() in EXTS:
                xs.append(preprocess(str(f)))
                ys.append(idx)
    print(f"{root.name}: {len(xs)} images")
    return np.stack(xs), np.array(ys)


def augmenter():
    # Figure 3.7: rotation, flip, zoom, brightness and contrast changes (training set only)
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.08),          # about +/- 30 degrees
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomBrightness(0.15, value_range=(-1.0, 1.0)),
        tf.keras.layers.RandomContrast(0.15),
    ], name="augmentation")


def build_model(weights="imagenet"):
    base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights=weights)
    base.trainable = False
    inputs = tf.keras.Input((IMG, IMG, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(len(CLASSES), activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="go_ananas_mobilenetv2"), base


def to_tflite(model, path: Path, quantize: bool = False):
    """Float32 export (about 9 MB) keeps every operator at version 1, so it runs on the
    TensorFlow Lite 2.16 runtime in the app. Set quantize=True for a smaller file (about 3 MB)
    only if the app's TensorFlow Lite library is upgraded to match the training version."""
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if quantize:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    path.write_bytes(conv.convert())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="dataset")
    ap.add_argument("--out", default="output")
    ap.add_argument("--epochs", type=int, default=25)
    ap.add_argument("--fine-tune-epochs", type=int, default=15)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--version", default=f"1.0-{date.today():%Y%m%d}")
    ap.add_argument("--copy-to-app", action="store_true", help="copy the model into app/src/main/assets")
    args = ap.parse_args()

    data, out = Path(args.data), Path(args.out)
    global CLASSES
    found = sorted(p.name for p in (data / "train").iterdir() if p.is_dir())
    if found != CLASSES:
        print("Using the class folders found in the dataset:", found)
        CLASSES = found
    out.mkdir(parents=True, exist_ok=True)
    x_tr, y_tr = load_split(data / "train")
    x_va, y_va = load_split(data / "val")
    x_te, y_te = load_split(data / "test")

    aug = augmenter()
    train_ds = (tf.data.Dataset.from_tensor_slices((x_tr, y_tr)).shuffle(len(x_tr), seed=42)
                .batch(args.batch).map(lambda x, y: (aug(x, training=True), y)).prefetch(2))
    val_ds = tf.data.Dataset.from_tensor_slices((x_va, y_va)).batch(args.batch)

    model, base = build_model()
    stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

    # Stage 1: train the new classifier head (learning rate 0.001)
    model.compile(tf.keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    counts = np.bincount(y_tr, minlength=len(CLASSES)).astype(float)
    weights = {i: float(counts.sum() / (len(CLASSES) * c)) for i, c in enumerate(counts) if c > 0}
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=args.epochs, callbacks=[stop], class_weight=weights)

    # Stage 2: fine-tune the last 30 layers of MobileNetV2 (learning rate 0.00001)
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    model.compile(tf.keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=args.fine_tune_epochs, callbacks=[stop], class_weight=weights)

    # History
    with open(out / "training_history.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["stage", "epoch", "accuracy", "loss", "val_accuracy", "val_loss"])
        for stage, h in (("head", h1), ("fine_tune", h2)):
            for i in range(len(h.history["loss"])):
                w.writerow([stage, i + 1] + [round(h.history[k][i], 4) for k in ("accuracy", "loss", "val_accuracy", "val_loss")])

    # Evaluation on the test set
    pred = model.predict(x_te, batch_size=args.batch).argmax(1)
    cm = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
    for t, p in zip(y_te, pred):
        cm[t, p] += 1
    acc = float((pred == y_te).mean())
    with open(out / "confusion_matrix.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["actual \\ predicted"] + CLASSES)
        for i, name in enumerate(CLASSES):
            w.writerow([name] + cm[i].tolist())
    lines = [f"{'class':<26}{'precision':>10}{'recall':>10}{'f1':>10}{'support':>10}"]
    for i, name in enumerate(CLASSES):
        tp = cm[i, i]
        prec = tp / cm[:, i].sum() if cm[:, i].sum() else 0
        rec = tp / cm[i].sum() if cm[i].sum() else 0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0
        lines.append(f"{name:<26}{prec:>10.4f}{rec:>10.4f}{f1:>10.4f}{cm[i].sum():>10}")
    lines.append(f"\nOverall accuracy: {acc:.4f} ({(pred == y_te).sum()} of {len(y_te)})")
    (out / "classification_report.txt").write_text("\n".join(lines))
    print("\n".join(lines))

    # Export
    model.save(out / "go_ananas_model.keras")
    to_tflite(model, out / "go_ananas_model.tflite")
    (out / "labels.txt").write_text("\n".join(CLASSES) + "\n")
    info = {"version": args.version, "trained": True, "test_accuracy": f"{acc * 100:.2f}%",
            "input": [IMG, IMG, 3], "classes": CLASSES}
    (out / "model_info.json").write_text(json.dumps(info, indent=2))

    if args.copy_to_app:
        assets = Path(__file__).resolve().parent.parent / "app" / "src" / "main" / "assets"
        assets.mkdir(parents=True, exist_ok=True)
        for name in ("go_ananas_model.tflite", "labels.txt", "model_info.json"):
            shutil.copy(out / name, assets / name)
        print(f"Copied model files to {assets}")


if __name__ == "__main__":
    main()

In [ ]:
%cd /content
!python split_dataset.py --src /content/go_ananas/dataset/all --dst /content/go_ananas/dataset
!python train_model.py --data /content/go_ananas/dataset --out /content/go_ananas/output

## 7. Save the results to Google Drive

In [ ]:
out = Path('/content/go_ananas/output')
for f in out.iterdir(): shutil.copy(f, DRIVE/f.name)
print(open(out/'classification_report.txt').read())
print('\nSaved to Google Drive / GoAnanas:', [f.name for f in out.iterdir()])
print('Upload go_ananas_model.tflite, labels.txt, and model_info.json to app/src/main/assets/ on GitHub.')
print('Paste the counts from dataset_summary.json into Table 3.2 and the results into Tables 4.1 to 4.3 of the paper.')